In [ ]:
import os
import random

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import functional as TF
from PIL import Image

In [ ]:
# ---------------------
# Grayscale Dataset with Augmentations
# ---------------------
class GrayscaleImageDataset(Dataset):
    def __init__(self, root_dir, transform=None, num_classes=3, add_noise=False, noise_std=0.1, pad_to=None, split="train",):
        self.root_dir = root_dir
        self.transform = transform
        self.num_classes = num_classes
        self.add_noise = add_noise
        self.noise_std = noise_std
        self.pad_to = pad_to  # Expected to be a tuple (height, width), e.g., (184, 184)
        self.split = split

        self.image_paths = []
        self.labels = []
        self.class_to_idx = {}

        for idx, class_name in enumerate(sorted(os.listdir(root_dir))):
            class_path = os.path.join(root_dir, class_name)
            if os.path.isdir(class_path):
                self.class_to_idx[class_name] = idx
                for img_name in os.listdir(class_path):
                    img_path = os.path.join(class_path, img_name)
                    self.image_paths.append(img_path)
                    self.labels.append(idx)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]

        # Load and convert to tensor
        image = Image.open(img_path).convert("L")
        image = transforms.ToTensor()(image)

        # PAD
        if self.pad_to:
            _, h, w = image.shape
            pad_h = max(0, self.pad_to[0] - h)
            pad_w = max(0, self.pad_to[1] - w)
            padding = (
                pad_w // 2,  # left
                pad_h // 2,  # top
                pad_w - pad_w // 2,  # right
                pad_h - pad_h // 2   # bottom
            )
            image = TF.pad(image, padding, fill=0)

        # Normalize to [-1, 1]
        image = transforms.Normalize(mean=[0.5], std=[0.5])(image)

         # === Augmentation SOLO su train ===
        if self.split == "train":
            if random.random() < 0.8:
                angle = random.uniform(-3, 3)
                translate = (random.uniform(-2, 2), random.uniform(-2, 2))
                image = TF.affine(image, angle=angle, translate=translate, scale=1.0, shear=0, fill=-1)

            if self.add_noise and torch.rand(1).item() < 0.6:
                noise = torch.randn_like(image) * self.noise_std
                image = torch.clamp(image + noise, -1.0, 1.0)

            #if random.random() < 0.5:
                #image = TF.gaussian_blur(image, kernel_size=3, sigma=random.uniform(0.5, 1.0))

        # One-hot encode label
        label_one_hot = torch.zeros(self.num_classes)
        label_one_hot[label] = 1

        return image, label_one_hot